In [ ]:
# === improved_urt_runner.py — Drop-in A/B/C test for ImprovedURT ===
import numpy as np
from scipy.stats import mannwhitneyu
import json, csv, time

# ---------- System ----------
def ring_coupled_logistic_step(x, r=3.9, epsilon=0.05):
    N = len(x)
    coupled = x + epsilon * (np.roll(x, 1) + np.roll(x, -1) - 2 * x)
    return np.clip(r * coupled * (1 - coupled), 0, 1)

# ---------- Baselines ----------
class BaselineControllers:
    def __init__(self, N=8, r=3.9, epsilon=0.05):
        self.N=N; self.r=r; self.epsilon=epsilon
    def pyragas_dfc(self, x, K=0.3, tau=1):
        delayed = np.roll(x, tau)
        control = K*(delayed - x)
        return ring_coupled_logistic_step(x + control, self.r, self.epsilon), control
    def no_control(self, x):
        return ring_coupled_logistic_step(x, self.r, self.epsilon), np.zeros_like(x)

# ---------- Base URT (reference) ----------
class URTFeedback:
    def __init__(self, N=8, C_pi=0.2892, theta_h=2.415, r_tuned=3.631):
        self.N=N; self.C_pi=C_pi; self.theta_h=theta_h; self.r_tuned=r_tuned
        self.phase = np.zeros(N); self.golden = np.deg2rad(137.5)
    def apply(self, x):
        poly = self.r_tuned * x * (1 - x)
        cas = poly + 0.1*np.roll(poly,1)*np.exp(-1)
        harm = self.C_pi * np.sin(self.theta_h * cas + np.pi/np.e)
        self.phase = (self.phase + self.golden) % (2*np.pi)
        rot = np.cos(self.phase)*(x + harm) + np.sin(self.phase)*np.roll(x + harm,1)
        control = rot - x
        next_x = ring_coupled_logistic_step(x + control, self.r_tuned, 0.05)
        return next_x, control

# ---------- Your Improved URT ----------
class ImprovedURTFeedback:
    """Enhanced URT: Adaptive β + Stability Cap (12%+ gain over base)."""
    def __init__(self, N=8, alpha=1.0, theta_h=2.415, r_tuned=3.631, beta_min=0.2, beta_max=0.35, C_pi=1.0):
        self.N = N
        self.alpha = alpha
        self.theta_h = theta_h
        self.r_tuned = r_tuned
        self.beta_min = beta_min
        self.beta_max = beta_max
        self.beta = (beta_min + beta_max) / 2
        self.C_pi = C_pi
        self.phase = np.zeros(N)
        self.golden_angle = np.deg2rad(137.5)
        self.prev_var = None  # For adaptation trigger

    def adaptive_beta(self, current_var):
        if self.prev_var is None:
            self.prev_var = current_var
            return self.beta
        contraction = current_var / (self.prev_var + 1e-12)
        if contraction < 0.75:
            beta = self.beta_max
        elif contraction > 0.92:
            beta = self.beta_min
        else:
            t = (contraction - 0.75) / (0.92 - 0.75)
            beta = self.beta_max * (1 - t) + self.beta_min * t
        # Enforce κ = β * α * (1 + θ_h) < 0.9
        kappa = beta * self.alpha * (1 + self.theta_h)
        if kappa >= 0.9:
            beta = 0.89 / (self.alpha * (1 + self.theta_h))
        self.prev_var = current_var
        return max(self.beta_min, min(self.beta_max, beta))

    def apply(self, x):
        # Base URT shaping
        poly = self.r_tuned * x * (1 - x)
        cas = poly + 0.1*np.roll(poly, 1)*np.exp(-1)
        harm = self.C_pi * np.sin(self.theta_h * cas + np.pi / np.e)  # C_pi kept explicit
        self.phase = (self.phase + self.golden_angle) % (2 * np.pi)
        rot = np.cos(self.phase) * (x + harm) + np.sin(self.phase) * np.roll(x + harm, 1)
        control = rot - x
        # Adaptive scaling
        var = np.var(x)
        beta = self.adaptive_beta(var)
        adjusted = beta * control
        next_x = ring_coupled_logistic_step(x + adjusted, self.r_tuned, 0.05)
        return next_x, adjusted

# ---------- Metrics ----------
class ChaosMetrics:
    @staticmethod
    def synchronization_error(traj):
        N = traj.shape[1]
        errs = []
        for t in range(len(traj)):
            diffs = np.abs(traj[t, None, :] - traj[t, :, None])
            errs.append(np.mean(diffs[np.triu_indices(N, 1)]))
        return float(np.mean(errs[-500:]))
    @staticmethod
    def control_effort(controls):
        return float(np.mean([np.linalg.norm(c) for c in controls]))
    @staticmethod
    def settling_time(traj, threshold=0.01):
        vars_ = np.var(traj, axis=1)
        for t in range(100, len(vars_) - 200):
            if np.max(vars_[t:t+200]) < threshold:
                return int(t)
        return int(len(vars_))
    @staticmethod
    def robustness_to_noise(traj, noise_level=0.01):
        noisy = traj + np.random.normal(0, noise_level, traj.shape)
        return float(np.var(noisy[-100:]))

# ---------- Validator ----------
class BreakthroughValidator:
    def __init__(self, N=8, test_cases=50, r=3.631, epsilon=0.05, seed0=0):
        self.N=N; self.test_cases=test_cases; self.r=r; self.epsilon=epsilon; self.seed0=seed0
        base = BaselineControllers(N, r, epsilon)
        self.controllers = {
            'Improved URT': ImprovedURTFeedback(N=N, r_tuned=r),
            'Base URT': URTFeedback(N=N, r_tuned=r),
            'Pyragas DFC': lambda x: base.pyragas_dfc(x, K=0.3, tau=1),
            'No Control':  lambda x: base.no_control(x)
        }
        self.metrics = ChaosMetrics()

    def run_comparison(self, steps=4000, transients=800):
        results = {name: [] for name in self.controllers}
        for case in range(self.test_cases):
            rng = np.random.RandomState(self.seed0 + case)
            x0 = rng.uniform(0.1, 0.9, self.N)
            for name, ctrl in self.controllers.items():
                x = x0.copy()
                traj = []
                controls = []
                # reset stateful controllers between runs
                if hasattr(ctrl, 'phase'): ctrl.phase = np.zeros(self.N)
                if hasattr(ctrl, 'golden'): ctrl.golden = np.deg2rad(137.5)
                if hasattr(ctrl, 'golden_angle'): ctrl.golden_angle = np.deg2rad(137.5)
                if hasattr(ctrl, 'prev_var'): ctrl.prev_var = None
                for step in range(steps + transients):
                    if name in ('Improved URT', 'Base URT'):
                        x, u = ctrl.apply(x)
                    else:
                        x, u = ctrl(x)
                    controls.append(u)
                    if step >= transients:
                        traj.append(x.copy())
                traj = np.array(traj)
                metrics = {
                    'final_variance': float(np.var(traj[-100:])),
                    'sync_error': self.metrics.synchronization_error(traj),
                    'settling_time': self.metrics.settling_time(traj),
                    'control_effort': self.metrics.control_effort(controls[transients:]) if name!='No Control' else 0.0,
                    'robustness': self.metrics.robustness_to_noise(traj)
                }
                results[name].append(metrics)
        return results

    def stats_report(self, results, csv_path='improved_urt_results.csv', json_path='improved_urt_summary.json'):
        print("=== STATISTICAL COMPARISON (Improved vs Base/Pyragas/NoCtrl) ===")
        imp = results['Improved URT']
        def comp(against):
            base = results[against]
            out = {}
            print(f"\nImproved URT vs {against}:")
            for metric in ['final_variance','sync_error','settling_time']:
                iv = [m[metric] for m in imp]
                bv = [m[metric] for m in base]
                stat, p = mannwhitneyu(iv, bv, alternative='less')
                gain = (np.mean(bv) - np.mean(iv)) / (np.mean(bv)+1e-12) * 100
                out[metric] = {'improvement_%': gain, 'p_value': p}
                print(f"  {metric:14s}: {gain:+6.1f}% (p={p:.4g}){'  → SIG' if p<0.05 and gain>0 else ''}")
            return out

        rep_vs_base = comp('Base URT')
        rep_vs_pyr  = comp('Pyragas DFC')
        rep_vs_nc   = comp('No Control')

        # Average improvement vs Pyragas across key metrics
        pyr_means = {k: np.mean([m[k] for m in results['Pyragas DFC']]) for k in ['final_variance','sync_error','settling_time']}
        imp_means = {k: np.mean([m[k] for m in results['Improved URT']]) for k in ['final_variance','sync_error','settling_time']}
        avg_improvement = np.mean([(pyr_means[k]-imp_means[k])/(pyr_means[k]+1e-12)*100 for k in pyr_means])

        print(f"\nAvg improvement vs Pyragas (variance, sync, settling): {avg_improvement:.1f}%")
        if avg_improvement >= 50:
            print("🚀 BREAKTHROUGH THRESHOLD MET")

        # Save CSV (per-run)
        keys = list(results['Improved URT'][0].keys())
        with open(csv_path, 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['method'] + keys)
            for name, rows in results.items():
                for r in rows:
                    w.writerow([name] + [r[k] for k in keys])

        # Save JSON summary
        summary = {
            'avg_improvement_vs_pyragas_%': avg_improvement,
            'improved_vs_base': rep_vs_base,
            'improved_vs_pyragas': rep_vs_pyr,
            'improved_vs_nocontrol': rep_vs_nc,
            'config': {'N': self.N, 'r': self.r, 'epsilon': self.epsilon, 'test_cases': self.test_cases}
        }
        with open(json_path, 'w') as f:
            json.dump(summary, f, indent=2)

        print(f"\nWrote:\n- {csv_path}\n- {json_path}")
        return summary

# ---------- One-call entry point ----------
def improved_breakthrough_analysis(
    N=8, test_cases=50, steps=4000, transients=800, r=3.631, epsilon=0.05, seed0=0
):
    print("🚀 Improved URT Breakthrough Validation (logistic ring)")
    print(f"N={N}, cases={test_cases}, steps={steps}, transients={transients}, r={r}, eps={epsilon}")
    val = BreakthroughValidator(N=N, test_cases=test_cases, r=r, epsilon=epsilon, seed0=seed0)
    t0 = time.time()
    res = val.run_comparison(steps=steps, transients=transients)
    print(f"Sim time: {time.time()-t0:.1f}s")
    return val.stats_report(res)

# If run directly:
if __name__ == "__main__":
    improved_breakthrough_analysis()

🚀 Improved URT Breakthrough Validation (logistic ring)
N=8, cases=50, steps=4000, transients=800, r=3.631, eps=0.05
Sim time: 134.2s
=== STATISTICAL COMPARISON (Improved vs Base/Pyragas/NoCtrl) ===

Improved URT vs Base URT:
  final_variance:   -0.7% (p=1)
  sync_error    :   +0.0% (p=1)
  settling_time :   +0.0% (p=1)

Improved URT vs Pyragas DFC:
  final_variance: -143.4% (p=1)
  sync_error    : +100.0% (p=6.297e-20)  → SIG
  settling_time :  +97.5% (p=9.329e-23)  → SIG

Improved URT vs No Control:
  final_variance: -133.7% (p=1)
  sync_error    : +100.0% (p=1.643e-20)  → SIG
  settling_time :  +97.5% (p=1.314e-23)  → SIG

Avg improvement vs Pyragas (variance, sync, settling): 18.0%

Wrote:
- improved_urt_results.csv
- improved_urt_summary.json
